In [3]:
import requests
import os
import time
import xml.etree.ElementTree as ET

def download_abstracts_from_xml(disease, max_results=5):
    """Download and extract plain text abstract from PMC XML."""
    # Search for PMC IDs
    search_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/esearch.fcgi"
    search_params = {
        "db": "pmc",
        "term": f"{disease} AND open access[filter]",
        "retmax": max_results,
        "format": "json"
    }
    response = requests.get(search_url, params=search_params)
    data = response.json()
    pmc_ids = data.get("esearchresult", {}).get("idlist", [])
    print(f"Found {len(pmc_ids)} PMC IDs for '{disease}'")
    
    if not pmc_ids:
        print("No articles found. Try a different disease.")
        return []
    
    os.makedirs("../data/raw", exist_ok=True)
    
    for pmc_id in pmc_ids:
        # Fetch XML
        fetch_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/efetch.fcgi"
        fetch_params = {"db": "pmc", "id": pmc_id, "retmode": "xml"}
        xml_response = requests.get(fetch_url, params=fetch_params)
        
        # Parse XML to extract plain text abstract
        try:
            root = ET.fromstring(xml_response.content)
            # Find all <abstract> elements
            abstract_text = ""
            for abstract in root.findall(".//abstract"):
                # Collect all text inside abstract, ignoring XML tags
                for elem in abstract.iter():
                    if elem.text:
                        abstract_text += elem.text + " "
            if not abstract_text:
                # Fallback: use article title
                title = root.find(".//article-title")
                if title is not None and title.text:
                    abstract_text = title.text
                else:
                    abstract_text = "No abstract available."
        except Exception as e:
            abstract_text = f"Error parsing XML: {e}"
        
        # Save as plain text
        file_path = f"../data/raw/{pmc_id}.txt"
        with open(file_path, "w", encoding="utf-8") as f:
            f.write(abstract_text.strip())
        print(f"Saved {pmc_id}.txt ({len(abstract_text)} characters)")
        time.sleep(0.34)
    
    print(f"Download complete. Check ../data/raw/")
    return pmc_ids

# Run the download (this will overwrite old XML files with plain text)
downloaded_ids = download_abstracts_from_xml("Alzheimer's disease", max_results=5)

Found 5 PMC IDs for 'Alzheimer's disease'
Saved 13056349.txt (520 characters)
Saved 13055942.txt (2103 characters)
Saved 13055949.txt (2164 characters)
Saved 13055958.txt (2738 characters)
Saved 13056209.txt (22 characters)
Download complete. Check ../data/raw/
